# Chapter 7: Evaluation & Benchmarks

## Complete Model Evaluation Notebook

This notebook walks through evaluating your trained language model step by step.

**Required files:**
- `best_model.pt` (your trained model from Chapter 6)
- `val.pt` (validation data)
- `train.pt` (optional, for baseline comparisons)

**Time:** ~10-15 minutes

**What you'll learn:**
- How to interpret perplexity
- Basic and advanced evaluation metrics
- Comparing against baselines
- Evaluating text generation quality
- Determining model readiness

## Setup: Install Packages and Import Libraries

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch transformers datasets numpy

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from collections import Counter, defaultdict
import json
import random
from typing import Dict, List, Optional

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("✅ Setup complete!")

Using device: cuda
✅ Setup complete!


## Upload Your Files

Run this cell to upload:
1. `best_model.pt` - Your trained model (required)
2. `val.pt` - Validation data (required)
3. `train.pt` - Training data (optional, for baseline comparisons)

In [ ]:
#you do no have to upload if you have already uploaded them manually!

from google.colab import files

print("📤 Upload your files:")
print("1. best_model.pt (required)")
print("2. val.pt (required)")
print("3. train.pt (optional)\n")

uploaded = files.upload()

# Verify files
print("\n" + "="*70)
required_files = ['best_model.pt', 'val.pt']
for f in required_files:
    if f in uploaded:
        print(f"✅ {f} uploaded successfully")
    else:
        print(f"❌ {f} is missing!")

if 'train.pt' in uploaded:
    print(f"✅ train.pt uploaded (baselines will be computed)")
else:
    print(f"⚠️  train.pt not uploaded (baselines will be skipped)")
print("="*70)

📤 Upload your files:
1. best_model.pt (required)
2. val.pt (required)
3. train.pt (optional)




❌ best_model.pt is missing!
❌ val.pt is missing!
⚠️  train.pt not uploaded (baselines will be skipped)


## Model Architecture

Define the model architecture (matches your training code).
No need to modify anything here!

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config['n_embd'] % config['n_head'] == 0
        self.n_head = config['n_head']
        self.n_embd = config['n_embd']
        self.head_size = config['n_embd'] // config['n_head']

        self.c_attn = nn.Linear(config['n_embd'], 3 * config['n_embd'])
        self.c_proj = nn.Linear(config['n_embd'], config['n_embd'])
        self.attn_dropout = nn.Dropout(config['dropout'])
        self.resid_dropout = nn.Dropout(config['dropout'])

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config['block_size'], config['block_size']))
                 .view(1, 1, config['block_size'], config['block_size'])
        )

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config['n_embd'], 4 * config['n_embd']),
            nn.GELU(),
            nn.Linear(4 * config['n_embd'], config['n_embd']),
            nn.Dropout(config['dropout']),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config['n_embd'])
        self.attn = MultiHeadAttention(config)
        self.ln2 = nn.LayerNorm(config['n_embd'])
        self.mlp = FeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class ConfigurableGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config['vocab_size'], config['n_embd'])
        self.position_embedding = nn.Embedding(config['block_size'], config['n_embd'])
        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config['n_layer'])
        ])
        self.ln_f = nn.LayerNorm(config['n_embd'])
        self.lm_head = nn.Linear(config['n_embd'], config['vocab_size'], bias=False)
        self.dropout = nn.Dropout(config['dropout'])

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = self.dropout(tok_emb + pos_emb)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1
            )
        return logits, loss


class ValTensorDataset(Dataset):
    def __init__(self, x2d: torch.Tensor):
        assert x2d.dim() == 2, "Expected a 2D tensor (N, T)"
        self.x2d = x2d.long()

    def __len__(self):
        return self.x2d.size(0)

    def __getitem__(self, idx):
        return self.x2d[idx]

print("✅ Model architecture loaded")

✅ Model architecture loaded


## Load Model and Data

In [ ]:
# Default config (will be overridden if model has saved config)
fallback_config = {
    'vocab_size': 50257,
    'n_embd': 512,
    'n_layer': 8,
    'n_head': 8,
    'block_size': 256,
    'dropout': 0.1,
    'batch_size': 32
}

# Load model checkpoint
print("Loading model...")
ckpt = torch.load('best_model.pt', map_location=device, weights_only=False)

if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
    state_dict = ckpt['model_state_dict']
    config = ckpt.get('config', fallback_config)
else:
    state_dict = ckpt
    config = fallback_config

model = ConfigurableGPT(config)
model.load_state_dict(state_dict, strict=False)
model.to(device).eval()

print(f"✅ Model loaded: {model.count_parameters()/1e6:.1f}M parameters")
print(f"   Config: {config}")

# Load validation data
print("\nLoading validation data...")
val_data = torch.load('val.pt', map_location='cpu')

if isinstance(val_data, torch.Tensor) and val_data.dim() == 2:
    val_dataset = ValTensorDataset(val_data)
else:
    raise ValueError("Unexpected val.pt format")

val_loader = DataLoader(
    val_dataset,
    batch_size=config.get('batch_size', 32),
    shuffle=False
)

print(f"✅ Validation data loaded: {len(val_dataset)} samples")

# Try to load training data for baseline comparisons
train_loader = None
try:
    print("\nLoading training data (for baselines)...")
    train_data = torch.load('train.pt', map_location='cpu')
    if isinstance(train_data, torch.Tensor) and train_data.dim() == 2:
        train_dataset = ValTensorDataset(train_data)
        train_loader = DataLoader(
            train_dataset,
            batch_size=config.get('batch_size', 32),
            shuffle=False
        )
        print(f"✅ Training data loaded: {len(train_dataset)} samples")
except:
    print("⚠️  train.pt not found - skipping baseline comparisons")

Loading model...
✅ Model loaded: 77.2M parameters
   Config: {'vocab_size': 50257, 'n_embd': 512, 'n_layer': 8, 'n_head': 8, 'block_size': 1024, 'dropout': 0.1}

Loading validation data...
✅ Validation data loaded: 672 samples

Loading training data (for baselines)...
✅ Training data loaded: 6050 samples


# Part 1: Understanding Perplexity

**Perplexity = exp(loss)**

It measures how "surprised" the model is by the data.
- Lower perplexity = better model
- Perplexity of 12 means the model is as confused as randomly choosing between 12 options

In [ ]:
def calculate_perplexity(loss):
    """Convert loss to perplexity"""
    return math.exp(loss)

def interpret_perplexity(perplexity, domain='general'):
    """Interpret perplexity score"""
    benchmarks = {
        'recipes': {'excellent': 15, 'good': 25, 'fair': 40},
        'general': {'excellent': 30, 'good': 50, 'fair': 80},
        'creative': {'excellent': 50, 'good': 80, 'fair': 120},
        'technical': {'excellent': 40, 'good': 70, 'fair': 100},
    }

    bench = benchmarks.get(domain, benchmarks['general'])

    if perplexity < bench['excellent']:
        return "Excellent - Ready for production"
    elif perplexity < bench['good']:
        return "Good - Acceptable for most uses"
    elif perplexity < bench['fair']:
        return "Fair - May need more training"
    else:
        return "Poor - Significant issues"

# Example: What does different loss values mean?
print("=" * 70)
print("UNDERSTANDING PERPLEXITY")
print("=" * 70)
print("\nLoss to Perplexity conversion:")
losses = [1.0, 2.0, 3.0, 4.0, 5.0]
for loss in losses:
    ppl = calculate_perplexity(loss)
    interpretation = interpret_perplexity(ppl)
    print(f"Loss: {loss:.2f} → Perplexity: {ppl:.2f} → {interpretation}")

UNDERSTANDING PERPLEXITY

Loss to Perplexity conversion:
Loss: 1.00 → Perplexity: 2.72 → Excellent - Ready for production
Loss: 2.00 → Perplexity: 7.39 → Excellent - Ready for production
Loss: 3.00 → Perplexity: 20.09 → Excellent - Ready for production
Loss: 4.00 → Perplexity: 54.60 → Fair - May need more training
Loss: 5.00 → Perplexity: 148.41 → Poor - Significant issues


# Part 2: Basic Evaluation

Computing:
- **Loss** (lower is better)
- **Perplexity** (lower is better)
- **Token Accuracy** (higher is better)

In [ ]:
@torch.no_grad()
def evaluate_basic(model, dataloader, device):
    """Basic evaluation: loss, perplexity, accuracy"""
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    for batch in dataloader:
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        x = x.to(device)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        # Next-token targets
        y = x.clone()
        y[:, :-1] = x[:, 1:]
        y[:, -1] = -1

        logits, loss = model(x, y)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)
        mask = (y != -1)
        correct = (preds[mask] == y[mask]).sum().item()
        total_correct += correct
        total_tokens += mask.sum().item()

    avg_loss = total_loss / max(len(dataloader), 1)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float('inf')
    accuracy = total_correct / total_tokens if total_tokens > 0 else 0.0

    return {
        'loss': avg_loss,
        'perplexity': perplexity,
        'accuracy': accuracy,
        'total_tokens': total_tokens,
    }

# Run basic evaluation
print("\n" + "=" * 70)
print("BASIC EVALUATION RESULTS")
print("=" * 70)

basic_results = evaluate_basic(model, val_loader, device)

print(f"\n📊 Loss: {basic_results['loss']:.4f}")
ppl_str = f"{basic_results['perplexity']:.2f}" if basic_results['perplexity'] != float('inf') else "∞"
print(f"📊 Perplexity: {ppl_str}")
print(f"📊 Token Accuracy: {basic_results['accuracy']:.2%}")
print(f"📊 Total Tokens Evaluated: {basic_results['total_tokens']:,}")

# Interpret results
interpretation = interpret_perplexity(basic_results['perplexity'], domain='general')
print(f"\n💡 Interpretation: {interpretation}")


BASIC EVALUATION RESULTS

📊 Loss: 4.7099
📊 Perplexity: 111.05
📊 Token Accuracy: 27.27%
📊 Total Tokens Evaluated: 687,456

💡 Interpretation: Poor - Significant issues


# Part 3: Advanced Metrics

Computing:
- **Token accuracy** - % of correctly predicted tokens
- **Sequence accuracy** - % of entire sequences predicted perfectly
- **Confidence analysis** - Accuracy when model is confident vs uncertain

In [ ]:
@torch.no_grad()
def evaluate_advanced(model, dataloader, device):
    """Advanced evaluation with confidence analysis"""
    model.eval()

    total_correct = 0
    total_tokens = 0
    perfect_sequences = 0
    total_sequences = 0

    high_conf_correct = 0
    high_conf_total = 0
    low_conf_correct = 0
    low_conf_total = 0

    confidence_threshold = 0.8

    for batch in dataloader:
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        x = x.to(device)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        y = x.clone()
        y[:, :-1] = x[:, 1:]
        y[:, -1] = -1

        logits, _ = model(x, y)
        probs = F.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)

        mask = (y != -1)
        correct_per_token = (preds == y) & mask

        total_correct += correct_per_token.sum().item()
        total_tokens += mask.sum().item()

        for i in range(x.size(0)):
            seq_mask = mask[i]
            if seq_mask.sum() > 0:
                if correct_per_token[i][seq_mask].all():
                    perfect_sequences += 1
                total_sequences += 1

        max_probs = probs.max(dim=-1)[0]
        high_conf_mask = (max_probs > confidence_threshold) & mask
        low_conf_mask = (max_probs <= confidence_threshold) & mask

        if high_conf_mask.any():
            high_conf_correct += correct_per_token[high_conf_mask].sum().item()
            high_conf_total += high_conf_mask.sum().item()

        if low_conf_mask.any():
            low_conf_correct += correct_per_token[low_conf_mask].sum().item()
            low_conf_total += low_conf_mask.sum().item()

    return {
        'token_accuracy': total_correct / max(total_tokens, 1),
        'sequence_accuracy': perfect_sequences / max(total_sequences, 1),
        'confidence_analysis': {
            'precision_high_confidence': high_conf_correct / max(high_conf_total, 1),
            'precision_low_confidence': low_conf_correct / max(low_conf_total, 1),
            'high_confidence_count': high_conf_total,
            'low_confidence_count': low_conf_total,
        }
    }

# Run advanced evaluation
print("\n" + "=" * 70)
print("ADVANCED EVALUATION RESULTS")
print("=" * 70)

advanced_results = evaluate_advanced(model, val_loader, device)

print(f"\n📊 Token Accuracy: {advanced_results['token_accuracy']:.2%}")
print(f"📊 Sequence Accuracy: {advanced_results['sequence_accuracy']:.2%}")
print(f"   (What % of entire sequences were predicted perfectly)")

print(f"\n🎯 Confidence Analysis:")
print(f"   When model is CONFIDENT (>80% probability):")
print(f"   - Accuracy: {advanced_results['confidence_analysis']['precision_high_confidence']:.2%}")
print(f"   - Count: {advanced_results['confidence_analysis']['high_confidence_count']:,} tokens")
print(f"\n   When model is UNCERTAIN (≤80% probability):")
print(f"   - Accuracy: {advanced_results['confidence_analysis']['precision_low_confidence']:.2%}")
print(f"   - Count: {advanced_results['confidence_analysis']['low_confidence_count']:,} tokens")

if advanced_results['confidence_analysis']['precision_high_confidence'] > 0.8:
    print(f"\n💡 When your model is confident, it's usually right! This is a good sign.")
else:
    print(f"\n💡 Your model still makes errors even when confident. Consider more training.")


ADVANCED EVALUATION RESULTS

📊 Token Accuracy: 27.27%
📊 Sequence Accuracy: 0.00%
   (What % of entire sequences were predicted perfectly)

🎯 Confidence Analysis:
   When model is CONFIDENT (>80% probability):
   - Accuracy: 85.42%
   - Count: 57,748 tokens

   When model is UNCERTAIN (≤80% probability):
   - Accuracy: 21.94%
   - Count: 629,708 tokens

💡 When your model is confident, it's usually right! This is a good sign.


# Part 4: Baseline Comparisons

Your model should be significantly better than:
1. **Random selection** - Randomly guessing tokens
2. **Most frequent token** - Always predicting the most common token
3. **3-gram model** - Simple n-gram language model

If not, something may be wrong with training!

In [ ]:
def random_baseline(dataloader, vocab_size):
    """Random prediction baseline"""
    correct = 0
    total = 0

    for batch in dataloader:
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        y = x.clone()
        y[:, :-1] = x[:, 1:]
        y[:, -1] = -1

        mask = (y != -1)
        random_preds = torch.randint(0, vocab_size, y.shape)
        correct += ((random_preds == y) & mask).sum().item()
        total += mask.sum().item()

    accuracy = correct / max(total, 1)
    return {'accuracy': accuracy, 'perplexity': vocab_size}


def most_frequent_baseline(train_loader, test_loader):
    """Always predict the most frequent token"""
    all_tokens = []
    for batch in train_loader:
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        all_tokens.extend(x.flatten().tolist())

    token_counts = Counter(all_tokens)
    most_frequent = token_counts.most_common(1)[0][0]

    correct = 0
    total = 0

    for batch in test_loader:
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        y = x.clone()
        y[:, :-1] = x[:, 1:]
        y[:, -1] = -1

        mask = (y != -1)
        correct += (y[mask] == most_frequent).sum().item()
        total += mask.sum().item()

    accuracy = correct / max(total, 1)
    return {'accuracy': accuracy, 'most_frequent_token': most_frequent}


class NgramModel:
    """Simple n-gram language model baseline"""

    def __init__(self, n=3):
        self.n = n
        self.counts = defaultdict(Counter)

    def train(self, dataloader):
        for batch in dataloader:
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            if len(x.shape) == 2:
                for seq in x:
                    self._train_sequence(seq.tolist())
            else:
                self._train_sequence(x.tolist())

    def _train_sequence(self, tokens):
        for i in range(len(tokens) - self.n):
            context = tuple(tokens[i:i+self.n-1])
            next_token = tokens[i+self.n-1]
            self.counts[context][next_token] += 1

    def predict(self, context):
        context = tuple(context[-(self.n-1):])
        if context in self.counts and len(self.counts[context]) > 0:
            return self.counts[context].most_common(1)[0][0]
        else:
            if len(context) > 1:
                return self.predict(list(context[1:]))
            else:
                return 0

    def evaluate(self, dataloader):
        correct = 0
        total = 0

        for batch in dataloader:
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            if len(x.shape) == 2:
                for seq in x:
                    c, t = self._evaluate_sequence(seq.tolist())
                    correct += c
                    total += t
            else:
                c, t = self._evaluate_sequence(x.tolist())
                correct += c
                total += t

        return correct / max(total, 1)

    def _evaluate_sequence(self, tokens):
        correct = 0
        total = 0

        for i in range(self.n-1, len(tokens)):
            context = tokens[:i]
            true_next = tokens[i]
            pred_next = self.predict(context)

            if pred_next == true_next:
                correct += 1
            total += 1

        return correct, total


# Only run if we have training data
if train_loader is not None:
    print("\n" + "=" * 70)
    print("BASELINE COMPARISONS")
    print("=" * 70)
    print("\nYour model should be MUCH better than these baselines!")
    print("-" * 70)

    # Baseline 1: Random
    print("\n1️⃣  Random Selection Baseline")
    random_results = random_baseline(val_loader, config['vocab_size'])
    print(f"   Accuracy: {random_results['accuracy']:.4%}")
    print(f"   Perplexity: ~{config['vocab_size']}")

    # Baseline 2: Most Frequent
    print("\n2️⃣  Most Frequent Token Baseline")
    freq_results = most_frequent_baseline(train_loader, val_loader)
    print(f"   Accuracy: {freq_results['accuracy']:.2%}")
    print(f"   (Always predicts token #{freq_results['most_frequent_token']})")

    # Baseline 3: N-gram
    print("\n3️⃣  3-gram Model Baseline")
    print("   Training 3-gram model... (this may take a minute)")
    ngram = NgramModel(n=3)
    ngram.train(train_loader)
    ngram_acc = ngram.evaluate(val_loader)
    print(f"   Accuracy: {ngram_acc:.2%}")

    # Comparison
    print("\n" + "=" * 70)
    print("COMPARISON SUMMARY")
    print("=" * 70)
    print(f"\nYour Model:     {basic_results['accuracy']:.2%}")
    print(f"Random:         {random_results['accuracy']:.4%}")
    print(f"Most Frequent:  {freq_results['accuracy']:.2%}")
    print(f"3-gram:         {ngram_acc:.2%}")

    best_baseline = max(random_results['accuracy'], freq_results['accuracy'], ngram_acc)

    if basic_results['accuracy'] > best_baseline:
        improvement = ((basic_results['accuracy'] - best_baseline) / best_baseline * 100)
        print(f"\n✅ SUCCESS! Your model is {improvement:.1f}% better than best baseline")
    else:
        print(f"\n⚠️  WARNING: Your model is not better than simple baselines!")
        print("   This suggests something may be wrong with training.")
else:
    print("\n⚠️  Skipping baseline comparisons (no train.pt file)")


BASELINE COMPARISONS

Your model should be MUCH better than these baselines!
----------------------------------------------------------------------

1️⃣  Random Selection Baseline
   Accuracy: 0.0017%
   Perplexity: ~50257

2️⃣  Most Frequent Token Baseline
   Accuracy: 4.16%
   (Always predicts token #198)

3️⃣  3-gram Model Baseline
   Training 3-gram model... (this may take a minute)
   Accuracy: 19.79%

COMPARISON SUMMARY

Your Model:     27.27%
Random:         0.0017%
Most Frequent:  4.16%
3-gram:         19.79%

✅ SUCCESS! Your model is 37.9% better than best baseline


# Part 5: Text Generation Quality

Generate sample texts and evaluate:
- **Repetition score** (lower is better) - How repetitive is the text?
- **Diversity score** (higher is better) - How varied is the vocabulary?

In [ ]:
@torch.no_grad()
def generate_text_simple(model, start_tokens, max_new_tokens=50, temperature=0.8, device='cuda'):
    """Simple text generation"""
    model.eval()
    tokens = start_tokens.clone().to(device)

    for _ in range(max_new_tokens):
        logits, _ = model(tokens.unsqueeze(0), None)
        logits = logits[0, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        tokens = torch.cat([tokens, next_token], dim=0)

        if tokens.size(0) >= model.config['block_size']:
            break

    return tokens


def calculate_repetition(tokens):
    """Measure repetitiveness using bigrams"""
    if len(tokens) < 2:
        return 0.0

    bigrams = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
    unique_bigrams = len(set(bigrams))
    total_bigrams = len(bigrams)

    repetition = 1.0 - (unique_bigrams / total_bigrams)
    return repetition


def calculate_diversity(tokens):
    """Measure vocabulary diversity"""
    if len(tokens) == 0:
        return 0.0

    unique_tokens = len(set(tokens))
    total_tokens = len(tokens)

    return unique_tokens / total_tokens


print("\n" + "=" * 70)
print("TEXT GENERATION QUALITY")
print("=" * 70)
print("\nGenerating sample texts and evaluating quality...")

# Generate multiple samples
num_samples = 5
start_tokens_list = []
for i in range(num_samples):
    sample_idx = random.randint(0, len(val_loader.dataset) - 1)
    start_seq = val_loader.dataset[sample_idx]
    start_tokens_list.append(start_seq[:10])

generation_results = []

for i, start_tokens in enumerate(start_tokens_list):
    generated = generate_text_simple(model, start_tokens, max_new_tokens=50, device=device)
    tokens_list = generated.tolist()

    repetition = calculate_repetition(tokens_list)
    diversity = calculate_diversity(tokens_list)

    generation_results.append({
        'sample_id': i + 1,
        'length': len(tokens_list),
        'repetition_score': repetition,
        'diversity_score': diversity,
        'tokens': tokens_list[:20]
    })

# Display results
avg_repetition = sum(r['repetition_score'] for r in generation_results) / len(generation_results)
avg_diversity = sum(r['diversity_score'] for r in generation_results) / len(generation_results)

print(f"\n📊 Average Repetition: {avg_repetition:.2%} (lower is better)")
print(f"📊 Average Diversity: {avg_diversity:.2%} (higher is better)")

print(f"\n💡 Quality Assessment:")
if avg_repetition < 0.2 and avg_diversity > 0.5:
    print("   ✅ Good generation quality - diverse and not repetitive")
elif avg_repetition < 0.3 or avg_diversity > 0.4:
    print("   ⚠️  Acceptable quality - some room for improvement")
else:
    print("   ❌ Poor quality - model may be overfitting or undertrained")


TEXT GENERATION QUALITY

Generating sample texts and evaluating quality...

📊 Average Repetition: 7.46% (lower is better)
📊 Average Diversity: 70.33% (higher is better)

💡 Quality Assessment:
   ✅ Good generation quality - diverse and not repetitive


# Part 6: Model Readiness Check

Is your model ready for deployment?

Define your requirements and check if the model meets them.

In [ ]:
class ModelReadinessChecker:
    """Determine if model is ready for deployment"""

    def __init__(self, requirements):
        self.requirements = requirements

    def check_readiness(self, basic_metrics, advanced_metrics=None):
        """Check if model meets requirements"""
        checks = {}

        if 'min_accuracy' in self.requirements:
            checks['accuracy'] = (
                basic_metrics.get('accuracy', 0) >= self.requirements['min_accuracy']
            )

        if 'max_perplexity' in self.requirements:
            checks['perplexity'] = (
                basic_metrics.get('perplexity', float('inf')) <=
                self.requirements['max_perplexity']
            )

        if advanced_metrics and 'min_sequence_accuracy' in self.requirements:
            checks['sequence_accuracy'] = (
                advanced_metrics.get('sequence_accuracy', 0) >=
                self.requirements['min_sequence_accuracy']
            )

        is_ready = all(checks.values()) if checks else False
        report = self._generate_report(checks, basic_metrics, advanced_metrics)

        return is_ready, report

    def _generate_report(self, checks, basic_metrics, advanced_metrics):
        """Generate detailed readiness report"""
        lines = []
        lines.append("\n" + "=" * 70)
        lines.append("MODEL READINESS REPORT")
        lines.append("=" * 70)

        if checks:
            lines.append("\nRequirement Checks:")
            for check_name, passed in checks.items():
                status = "✅ PASS" if passed else "❌ FAIL"
                lines.append(f"  {check_name}: {status}")

        lines.append("\nCurrent Metrics:")
        if 'accuracy' in basic_metrics:
            lines.append(f"  Accuracy: {basic_metrics['accuracy']:.2%}")
        if 'perplexity' in basic_metrics:
            ppl = basic_metrics['perplexity']
            ppl_str = f"{ppl:.2f}" if ppl != float('inf') else "∞"
            lines.append(f"  Perplexity: {ppl_str}")

        if advanced_metrics and 'sequence_accuracy' in advanced_metrics:
            lines.append(f"  Sequence Accuracy: {advanced_metrics['sequence_accuracy']:.2%}")

        lines.append("\nRecommendation:")
        if checks and all(checks.values()):
            lines.append("  ✅ Model is READY for deployment!")
        else:
            lines.append("  ❌ Model needs MORE TRAINING")
            if checks:
                failed_checks = [k for k, v in checks.items() if not v]
                lines.append(f"  Failed checks: {', '.join(failed_checks)}")

        lines.append("=" * 70)

        return '\n'.join(lines)


# Define your requirements (customize these!)
requirements = {
    'min_accuracy': 0.30,           # Minimum token accuracy
    'max_perplexity': 150,          # Maximum perplexity
    'min_sequence_accuracy': 0.05,  # Minimum sequence accuracy
}

print("\n" + "=" * 70)
print("MODEL READINESS CHECK")
print("=" * 70)
print("\nYour requirements:")
for req, val in requirements.items():
    print(f"  {req}: {val}")

checker = ModelReadinessChecker(requirements)
is_ready, report = checker.check_readiness(basic_results, advanced_results)

print(report)


MODEL READINESS CHECK

Your requirements:
  min_accuracy: 0.3
  max_perplexity: 150
  min_sequence_accuracy: 0.05

MODEL READINESS REPORT

Requirement Checks:
  accuracy: ❌ FAIL
  perplexity: ✅ PASS
  sequence_accuracy: ❌ FAIL

Current Metrics:
  Accuracy: 27.27%
  Perplexity: 111.05
  Sequence Accuracy: 0.00%

Recommendation:
  ❌ Model needs MORE TRAINING
  Failed checks: accuracy, sequence_accuracy


# Part 7: Save Complete Results

Save everything to a JSON file for your records!

In [ ]:
def make_serializable(obj):
    """Convert objects to JSON-serializable format"""
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_serializable(item) for item in obj]
    elif isinstance(obj, (float, int)):
        if math.isinf(obj) or math.isnan(obj):
            return str(obj)
        return float(obj)
    else:
        return obj

all_results = {
    'basic_metrics': basic_results,
    'advanced_metrics': advanced_results,
    'generation_quality': {
        'average_repetition': avg_repetition,
        'average_diversity': avg_diversity,
        'samples': generation_results
    },
    'model_config': config,
    'is_ready_for_deployment': is_ready,
}

# Add baseline results if available
if train_loader is not None:
    all_results['baseline_comparisons'] = {
        'random': random_results,
        'most_frequent': freq_results,
        'ngram_3': {'accuracy': ngram_acc}
    }

# Save to JSON
results_serializable = make_serializable(all_results)

with open('chapter7_evaluation_results.json', 'w') as f:
    json.dump(results_serializable, f, indent=2)

print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)
print("\n✅ Complete evaluation results saved to: chapter7_evaluation_results.json")

# Download the results
print("\nDownloading results file...")
files.download('chapter7_evaluation_results.json')


RESULTS SAVED

✅ Complete evaluation results saved to: chapter7_evaluation_results.json



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Summary & Next Steps

## 📊 Your Model Performance Summary

In [ ]:
print("\n" + "=" * 70)
print("CHAPTER 7 COMPLETE! 🎉")
print("=" * 70)

print("\n📊 Summary of Your Model:")
print(f"  • Perplexity: {basic_results['perplexity']:.2f}")
print(f"  • Token Accuracy: {basic_results['accuracy']:.2%}")
print(f"  • Sequence Accuracy: {advanced_results['sequence_accuracy']:.2%}")
print(f"  • Generation Quality: Repetition {avg_repetition:.2%}, Diversity {avg_diversity:.2%}")

if is_ready:
    print(f"\n✅ Your model PASSED the readiness check!")
else:
    print(f"\n⚠️  Your model needs more training to meet requirements.")

print("\n📚 What You Learned:")
print("  ✓ How to interpret perplexity")
print("  ✓ Basic and advanced evaluation metrics")
print("  ✓ Comparing against baselines")
print("  ✓ Evaluating text generation quality")
print("  ✓ Determining model readiness")

print("\n🎯 Next Steps:")
print("  1. Review your evaluation results")
print("  2. If not ready: adjust training (more data, longer training, tune hyperparameters)")
print("  3. If ready: proceed to Chapter 8 (Advanced Fine-Tuning with LoRA)")

print("\n" + "=" * 70)


CHAPTER 7 COMPLETE! 🎉

📊 Summary of Your Model:
  • Perplexity: 111.05
  • Token Accuracy: 27.27%
  • Sequence Accuracy: 0.00%
  • Generation Quality: Repetition 7.46%, Diversity 70.33%

⚠️  Your model needs more training to meet requirements.

📚 What You Learned:
  ✓ How to interpret perplexity
  ✓ Basic and advanced evaluation metrics
  ✓ Comparing against baselines
  ✓ Evaluating text generation quality
  ✓ Determining model readiness

🎯 Next Steps:
  1. Review your evaluation results
  2. If not ready: adjust training (more data, longer training, tune hyperparameters)
  3. If ready: proceed to Chapter 8 (Advanced Fine-Tuning with LoRA)

